# Coordination Number Variants

Beyond the simple integer coordination number (CN), pyscal provides
several refined coordination descriptors:

| Descriptor | Function | Description |
|------------|----------|-------------|
| CN | `coordination_number` | Integer neighbor count |
| ECoN | `effective_coordination_number` | Distance-weighted (Hoppe 1979) |
| GCN | `generalized_coordination_number` | Accounts for neighbor coordination (Calle-Vallejo 2015) |
| Local density | `local_density` | Atoms per unit volume from neighbor distance |

In [ ]:
import pyscal
from pyscal.structures import make_crystal
import numpy as np

## Perfect Crystals

In [ ]:
print("{:<6} {:>4} {:>8} {:>8} {:>10}".format("Struct", "CN", "ECoN", "GCN", "Density"))
print("-" * 40)

for name, lc in [("fcc", 4.05), ("bcc", 2.87), ("hcp", 3.21)]:
    atoms = make_crystal(name, lattice_constant=lc, repetitions=(4,4,4))
    pyscal.find_neighbors(atoms, method="cutoff", cutoff=0)
    cn   = pyscal.coordination_number(atoms)
    econ = pyscal.effective_coordination_number(atoms)
    gcn  = pyscal.generalized_coordination_number(atoms)
    rho  = pyscal.local_density(atoms)
    print(f"{name:<6} {cn[0]:>4d} {econ[0]:>8.2f} {gcn[0]:>8.2f} {rho[0]:>10.6f}")

## Effective Coordination Number (ECoN)

ECoN (Hoppe, 1979) weights each neighbor by a continuous function of
distance. For BCC, the 6 second-shell neighbors receive reduced weight,
giving ECoN < 14.

In [ ]:
bcc = make_crystal("bcc", lattice_constant=2.87, repetitions=(4,4,4))
pyscal.find_neighbors(bcc, method="cutoff", cutoff=0)
cn = pyscal.coordination_number(bcc)
econ = pyscal.effective_coordination_number(bcc)
print(f"BCC: CN = {cn[0]},  ECoN = {econ[0]:.4f}")
print(f"ECoN < CN because second-shell neighbors are down-weighted")

## Generalized Coordination Number (GCN)

GCN (Calle-Vallejo *et al.*, 2015) penalizes under-coordinated
neighbors, useful for surface catalysis studies.

In [ ]:
atoms = make_crystal("fcc", lattice_constant=4.05, repetitions=(4,4,4))
pyscal.find_neighbors(atoms, method="cutoff", cutoff=0)
gcn = pyscal.generalized_coordination_number(atoms)
print(f"Bulk FCC GCN = {gcn[0]:.2f} (= CN for fully coordinated bulk)")
print(f"All atoms identical: std = {gcn.std():.2e}")

## Accessing Results

All values are stored in `atoms.arrays`:

In [ ]:
atoms = make_crystal("fcc", lattice_constant=4.05, repetitions=(3,3,3))
pyscal.find_neighbors(atoms, method="cutoff", cutoff=0)
pyscal.coordination_number(atoms)
pyscal.effective_coordination_number(atoms)
pyscal.generalized_coordination_number(atoms)
pyscal.local_density(atoms)

keys = [k for k in atoms.arrays if k.startswith("pyscal_") and
        any(x in k for x in ["cn", "econ", "gcn", "density"])]
for k in sorted(keys):
    print(f"{k}: shape={atoms.arrays[k].shape}, first={atoms.arrays[k][0]:.4f}")

## References

1. R. Hoppe, *Z. Kristallogr.* **150**, 23 (1979). (ECoN)
2. F. Calle-Vallejo *et al.*, *Science* **350**, 185 (2015). [doi:10.1126/science.aab3501](https://doi.org/10.1126/science.aab3501) (GCN)